# Stage 6 — Clean Bangla Legal Answer Generation (Kaggle-ready)

এই notebook আগের debugging/checkpoint ফাইলের ওপর নির্ভর করে না। এটি **নতুন output filename** ব্যবহার করে, যাতে `gold_qa_set_v2.csv` overwrite না হয়।

## আগে যা লাগবে
Kaggle Notebook → **Settings → Accelerator → GPU** (GPU না থাকলেও CPU-তে চলতে পারে, তবে ধীর হবে)।

`Add Input` থেকে এই ৩টি file-সহ dataset attach করো:
- `gold_qa_set_v2.csv`
- `stage5_context_dense_top20.csv`
- `bdlegalrag_corpus_v1.json`

Notebook-টি এক cell করে চালাবে। প্রথমে 5-question pilot হবে। Pilot-এর উত্তর evidence-এর সঙ্গে মিলিয়ে দেখে তবেই full run চালাবে।

**গুরুত্বপূর্ণ:** কোনো notebook legal answer-এর correctness নিশ্চিত করতে পারে না। Generated answers-কে research evaluation-এর আগে manually/independently যাচাই করতে হবে।


In [1]:
# Cell 1 — Environment check
import os, sys, glob, json, re, time, subprocess, importlib.util
from pathlib import Path

print("Python:", sys.version)
print("Working directory:", os.getcwd())
print("GPU available before setup:", importlib.util.find_spec("torch") is not None)


Python: 3.13.16 (main, Oct  1 2026, 15:40:24) [GCC 13.3.0]
Working directory: /kaggle/working
GPU available before setup: True


In [2]:
# # Cell 2 — Install compatible libraries
# # Run this cell once in a fresh Kaggle session, then restart session if it installs/changes packages.
# %pip -q install "transformers==4.51.3" "accelerate>=1.0.0" sentencepiece


In [3]:
# Cell 2 — Install compatible libraries with 4-bit support
%pip -q install "transformers==4.51.3" "accelerate>=1.0.0" bitsandbytes sentencepiece

Note: you may need to restart the kernel to use updated packages.


In [4]:
# Cell 3 — Restart the Kaggle session after Cell 2 if packages were installed/changed.
# After restart, continue from Cell 4. Do not re-run the install cell unless necessary.
import torch, transformers, pandas as pd
print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


PyTorch: 2.11.0+cu130
Transformers: 4.51.3
CUDA available: True
GPU: Tesla T4


In [5]:
# Cell 4 — Find required inputs (recursively searches /kaggle/input and /kaggle/working)
import glob
from pathlib import Path

def find_input(filename):
    candidates = [
        f"/kaggle/input/**/{filename}",
        f"/kaggle/working/{filename}",
        filename,
    ]
    found = []
    for pattern in candidates:
        found.extend(glob.glob(pattern, recursive=True))
    # Deduplicate while preserving order
    found = list(dict.fromkeys(p for p in found if Path(p).is_file()))
    if not found:
        raise FileNotFoundError(
            f"Missing {filename}. In Kaggle, click Add Input and attach the dataset containing this file."
        )
    # Prefer /kaggle/input if available
    found.sort(key=lambda p: (not p.startswith("/kaggle/input/"), len(p)))
    return found[0]

QA_PATH = find_input("gold_qa_set_v2.csv")
TOP20_PATH = find_input("stage5_context_dense_top20.csv")
CORPUS_PATH = find_input("bdlegalrag_corpus_v1.json")

print("QA:", QA_PATH)
print("Top-20 retrieval:", TOP20_PATH)
print("Corpus:", CORPUS_PATH)


QA: /kaggle/input/datasets/tanmimalamnuhin/qa-verification-updated-gold-qa-v2/gold_qa_set_v2.csv
Top-20 retrieval: /kaggle/input/datasets/tanmimalamnuhin/qa-verification-updated-gold-qa-v2/stage5_context_dense_top20.csv
Corpus: /kaggle/input/datasets/tanmimalamnuhin/qa-verification-updated-gold-qa-v2/bdlegalrag_corpus_v1.json


In [6]:
# Cell 5 — Load and validate all input data
qa = pd.read_csv(QA_PATH, encoding="utf-8-sig")
top20 = pd.read_csv(TOP20_PATH, encoding="utf-8-sig")
with open(CORPUS_PATH, "r", encoding="utf-8") as f:
    corpus = json.load(f)

required_qa = {"qa_id", "question"}
required_top20 = {"qa_id", "rank", "corpus_idx"}
assert required_qa.issubset(qa.columns), f"QA file missing columns: {required_qa - set(qa.columns)}"
assert required_top20.issubset(top20.columns), f"Top20 file missing columns: {required_top20 - set(top20.columns)}"
assert "acts" in corpus and isinstance(corpus["acts"], list), "Corpus JSON must contain an 'acts' list."
assert qa["qa_id"].notna().all(), "QA file has missing qa_id."
assert qa["qa_id"].astype(str).is_unique, "QA file has duplicate qa_id."
assert qa["question"].notna().all(), "QA file has missing question."

corpus_sections = []
for act in corpus["acts"]:
    for sec in act.get("sections", []):
        corpus_sections.append({
            "act_id": act.get("act_id"),
            "act_title": act.get("act_title", ""),
            "act_no": act.get("act_no", ""),
            "act_year": act.get("act_year", ""),
            "section_number": sec.get("section_number", ""),
            "section_text": sec.get("section_text", ""),
            "language": sec.get("language", "")
        })
corpus_df = pd.DataFrame(corpus_sections)
assert len(corpus_df) > 0, "No sections found in corpus."

top20["qa_id"] = top20["qa_id"].astype(str)
qa["qa_id"] = qa["qa_id"].astype(str)
top20["rank"] = pd.to_numeric(top20["rank"], errors="coerce")
top20["corpus_idx"] = pd.to_numeric(top20["corpus_idx"], errors="coerce")
assert top20["rank"].notna().all(), "Top20 has invalid rank values."
assert top20["corpus_idx"].notna().all(), "Top20 has invalid corpus_idx values."
assert top20["corpus_idx"].between(0, len(corpus_df)-1).all(), "Some corpus_idx values are out of range."

qa_ids = set(qa["qa_id"])
retrieval_ids = set(top20["qa_id"])
missing_retrieval = sorted(qa_ids - retrieval_ids)
print("QA rows:", len(qa), "| unique IDs:", qa["qa_id"].nunique())
print("Top20 rows:", len(top20), "| corpus sections:", len(corpus_df))
print("Missing retrieval results for QA IDs:", len(missing_retrieval))
print("Invalid corpus indices: 0 (validated)")
if missing_retrieval:
    print("First missing IDs:", missing_retrieval[:10])


QA rows: 180 | unique IDs: 180
Top20 rows: 3600 | corpus sections: 1575
Missing retrieval results for QA IDs: 0
Invalid corpus indices: 0 (validated)


In [7]:
# Cell 6 — Build Clean Top-3 Contexts
def clean_text(value):
    if pd.isna(value): return ""
    return re.sub(r"\s+", " ", str(value)).strip()

# Take Top-3 retrieved passages (Reduces noise significantly for 3B models)
top3 = top20.sort_values(["qa_id", "rank"]).groupby("qa_id", sort=False).head(3).copy()

context_rows = []
for qa_id, group in top3.groupby("qa_id", sort=False):
    parts = []
    seen = set()
    for _, row in group.sort_values("rank").iterrows():
        idx = int(row["corpus_idx"])
        sec_row = corpus_df.iloc[idx]
        title = clean_text(sec_row["act_title"])
        sec_num = clean_text(sec_row["section_number"])
        body = clean_text(sec_row["section_text"])
        
        if (title, sec_num) in seen: continue
        seen.add((title, sec_num))
        parts.append(f"[আইন: {title} | ধারা: {sec_num}]\nবিধান: {body}")
        
    context_rows.append({"qa_id": str(qa_id), "retrieved_context": "\n\n".join(parts)})

context_df = pd.DataFrame(context_rows)
base = qa[["qa_id", "question", "act_title", "section_number", "final_answer"]].copy()
base = base.merge(context_df, on="qa_id", how="left")
base["retrieved_context"] = base["retrieved_context"].fillna("")
print("✓ Top-3 Context Packaging Complete. Ready for clean generation.")

✓ Top-3 Context Packaging Complete. Ready for clean generation.


In [8]:
# # Cell 7 — Load Official Google Gemma-2 from Kaggle
# from transformers import AutoTokenizer, AutoModelForCausalLM
# import glob, torch

# # Find the real base model directory containing config.json
# configs = glob.glob("/kaggle/input/**/config.json", recursive=True)
# base_model_dir = None
# for c in configs:
#     # Filter out adapter configs, only keep real model config
#     if "adapter" not in c and "gemma" in c.lower():
#         base_model_dir = os.path.dirname(c)
#         break

# if not base_model_dir:
#     raise FileNotFoundError("আসল Gemma-2 বেস মডেল পাওয়া যায়নি। নিশ্চিত করুন গুগলের অফিশিয়াল gemma-2 (~5GB) যুক্ত করা হয়েছে।")

# print(f"✓ Loading official Google Gemma-2 from: {base_model_dir}")

# tokenizer = AutoTokenizer.from_pretrained(base_model_dir)
# model = AutoModelForCausalLM.from_pretrained(
#     base_model_dir,
#     torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
#     device_map="auto"
# )
# model.eval()

# # Suppress generation warnings
# model.generation_config.do_sample = False
# model.generation_config.temperature = None
# model.generation_config.top_p = None
# model.generation_config.top_k = None

# if tokenizer.pad_token_id is None:
#     tokenizer.pad_token = tokenizer.eos_token

# print(f"✓ Google Gemma-2 loaded successfully!")

In [9]:
# Cell 7 — Load Qwen2.5-7B-Instruct in 4-bit (SOTA 7B Model)
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch

MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"

# 4-bit Quantization Config (Takes only ~5.5 GB VRAM on T4!)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4"
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto"
)
model.eval()

# Clean generation warnings
model.generation_config.do_sample = False
model.generation_config.temperature = None
model.generation_config.top_p = None
model.generation_config.top_k = None

if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"✓ Model Loaded: {MODEL_NAME} in 4-bit!")
print(f"✓ VRAM Allocated: {torch.cuda.memory_allocated(0)/(1024**3):.2f} GB / {torch.cuda.get_device_properties(0).total_memory/(1024**3):.2f} GB")

I0000 00:00:1793845732.561367     184 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

model-00003-of-00004.safetensors:   0%|          | 0.00/3.86G [00:00<?, ?B/s]

model-00002-of-00004.safetensors:   0%|          | 0.00/3.86G [00:00<?, ?B/s]

model-00001-of-00004.safetensors:   0%|          | 0.00/3.95G [00:00<?, ?B/s]

model-00004-of-00004.safetensors:   0%|          | 0.00/3.56G [00:00<?, ?B/s]

Sliding Window Attention is enabled but not implemented for `sdpa`; unexpected results may be encountered.


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

✓ Model Loaded: Qwen/Qwen2.5-7B-Instruct in 4-bit!
✓ VRAM Allocated: 1.87 GB / 14.56 GB


In [10]:
# Cell 8 — Full Clean Generation Engine for Google Gemma-2 (No System Role Error)
import re

def clean_answer(text):
    """Trims trailing incomplete words back to the last complete Bengali sentence."""
    if not text: return ""
    text = text.strip()
    
    # If the model tried to repeat a second 'আইনের নাম:' block, keep only the first
    second_act = text.find('আইনের নাম:', 15)
    if second_act != -1:
        text = text[:second_act].strip()
        
    text = re.sub(r'[\s]+$', '', text)
    
    # Find the last sentence-ending punctuation (দাঁড়ি '।' or question mark '?' or exclamation '!')
    last_punc = max(text.rfind('।'), text.rfind('?'), text.rfind('!'))
    if last_punc != -1 and last_punc > 30:
        return text[:last_punc + 1].strip()
        
    return text

def make_rag_prompt(question, context):
    return f"""তুমি একজন দক্ষ আইনি পরামর্শক। নিচে দেওয়া আইনগত প্রমাণের ওপর ভিত্তি করে প্রশ্নের সঠিক ও স্পষ্ট উত্তর দাও।

কঠোর নির্দেশনা:
১. আইনি ধারা হুবহু কপি করবে না; প্রমাণ পড়ে নিজের ভাষায় প্রশ্নের সরাসরি উত্তর বুঝিয়ে লেখো।
২. উত্তরটি সর্বোচ্চ ৩ থেকে ৪টি পূর্ণ বাক্যে সমাপ্ত করবে।
৩. একই কথা বারবার লিখবে না বা অপ্রয়োজনীয় অতিরিক্ত আইনের আলোচনা করবে না।
৪. উত্তর শেষ হলে দাঁড়ি (।) দিয়ে থেমে যাবে।

উত্তরের ফরম্যাট:
আইনের নাম: [আইনের পুরো নাম]
ধারা: [ধারা নম্বর]
উত্তর: [প্রশ্নের স্পষ্ট ও পূর্ণাঙ্গ আইনি উত্তর]

আইনগত প্রমাণ:
{context}

প্রশ্ন: {question}

উত্তর:"""

def generate_response(question, context=None, mode="rag", max_new_tokens=400):
    if not str(context).strip():
        return "প্রদত্ত প্রমাণ পাওয়া যায়নি; নিশ্চিত উত্তর দেওয়া সম্ভব নয়।"
    
    system_instruction = "তুমি বাংলাদেশের আইনের সহায়ক। প্রদত্ত প্রমাণের ভিত্তিতে প্রশ্নের সরাসরি, পূর্ণাঙ্গ ও সহজ বাংলায় উত্তর দাও।"
    user_content = make_rag_prompt(str(question), str(context))
    
    # Merge system instruction into user message for Gemma compatibility:
    full_user_prompt = f"{system_instruction}\n\n{user_content}"
    
    messages = [
        {"role": "user", "content": full_user_prompt}
    ]
    
    rendered = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    tokenizer.truncation_side = "left"
    inputs = tokenizer(rendered, return_tensors="pt", truncation=True, max_length=3500)
    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            repetition_penalty=1.12,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.pad_token_id
        )
        
    new_tokens = output[0][inputs["input_ids"].shape[1]:]
    raw_answer = tokenizer.decode(new_tokens, skip_special_tokens=True, clean_up_tokenization_spaces=False).strip()
    
    # Automatically clean and trim back to the last complete sentence!
    final_answer = clean_answer(raw_answer)
    return final_answer

print("✓ Gemma-2 Compatible Generation Engine Configured Successfully.")

✓ Gemma-2 Compatible Generation Engine Configured Successfully.


In [11]:
# Cell 9 — 5-Question Pilot for Qwen2.5-7B (RAG Mode)
pilot_subset = base.head(5).copy()
pilot_records = []

print("Running 5-Question Pilot on Qwen2.5-7B...")
for i, (_, row) in enumerate(pilot_subset.iterrows(), start=1):
    q_id = row["qa_id"]
    q_text = row["question"]
    ctx = row["retrieved_context"]
    gold_act = row["act_title"]
    gold_sec = str(row["section_number"])
    
    rag_ans = generate_response(q_text, context=ctx, mode="rag")
    
    pilot_records.append({
        "qa_id": q_id,
        "question": q_text,
        "gold_act": gold_act,
        "gold_sec": gold_sec,
        "qwen7b_answer": rag_ans
    })
    print(f"✓ Completed {i}/5: {q_id}")

pilot_df = pd.DataFrame(pilot_records)
pilot_df.to_csv("/kaggle/working/qwen7b_pilot_5_answers.csv", index=False, encoding="utf-8-sig")

for _, r in pilot_df.iterrows():
    print("="*80)
    print(f"ID: {r['qa_id']} | TARGET: {r['gold_act']} (ধারা {r['gold_sec']})")
    print(f"QUESTION: {r['question']}")
    print(f"--- [Qwen2.5-7B Answer] ---\n{r['qwen7b_answer']}")

Running 5-Question Pilot on Qwen2.5-7B...
✓ Completed 1/5: qa_0001
✓ Completed 2/5: qa_0002
✓ Completed 3/5: qa_0003
✓ Completed 4/5: qa_0004
✓ Completed 5/5: qa_0005
ID: qa_0001 | TARGET: ভূমি-খতিয়ান (পার্বত্য চট্টগ্রাম) অধ্যাদেশ, ১৯৮৪ (ধারা 12)
QUESTION: পাহাড়ি এলাকার জমির খতিয়ানে যদি ভুল বা কোনো ধোঁকাবাজি করে তথ্য ঢুকানো হয়, খতিয়ান চূড়ান্ত হওয়ার আগে কি তা ঠিক করার কোনো উপায় আছে?
--- [Qwen2.5-7B Answer] ---
আইনের নাম: ভূমি-খতিয়ান (পার্বত্য চট্টগ্রাম) অধ্যাদেশ, ১৯৮৪
ধারা: 12
উত্তর: সেটেলমেন্ট অফিসার যদি বিবেচনায় ভূমি-খতিয়ানে প্রতারণার মাধ্যমে কোনো তথ্য লিপিবদ্ধ করা হয়, তাহলে তিনি তার সংশোধন করার জন্য নির্দেশ দিতে পারেন। কিন্তু এই নির্দেশ দিবার আগে সংশোধিত পক্ষগণকে শুনানীর জন্য যুক্তিসঙ্গত সুযোগ প্রদান করতে হবে। চূড়ান্ত প্রকাশনের পর খতিয়ানটি শুদ্ধ হিসাবে গণ্য হবে।
ID: qa_0002 | TARGET: অস্থাবর সম্পত্তি হুকুমদখল আইন, ১৯৮৮ (ধারা 3)
QUESTION: সরকারের কোনো কাজের জন্য হঠাৎ করে আমার কোনো জিনিসপত্র কি সাময়িকভাবে নিয়ে নিতে পারে, আর নিলে আমাকে কীভাবে জানাবে?
--- [Qwen2.5-7B Answer] ---
আই

## Gate before full generation
Review the 5 pilot answers above. If they are irrelevant, truncate, contain `�`, or change a legal number/condition, **do not run the full generation yet**. Adjust the prompt/model setup and repeat the pilot. A clean character check alone is not enough to establish legal accuracy.

When the pilot is acceptable for your next experiment, run Cell 10.


In [12]:
# # Cell 10 — Full generation with resumable checkpoint
# # This is a fresh output/checkpoint name; it does NOT read old checkpoints or overwrite the gold QA file.
# from pathlib import Path
# import pandas as pd
# import time

# CHECKPOINT_PATH = "/kaggle/working/stage6_clean_generation_checkpoint.csv"
# OUTPUT_PATH = "/kaggle/working/stage6_generated_answers_clean_v1.csv"

# # Start from scratch only if this new checkpoint does not exist.
# if Path(CHECKPOINT_PATH).exists():
#     checkpoint = pd.read_csv(CHECKPOINT_PATH, encoding="utf-8-sig", dtype={"qa_id": str})
#     results = checkpoint.drop_duplicates("qa_id", keep="last").to_dict("records")
#     done_ids = set(checkpoint["qa_id"].astype(str))
#     print("Resuming from NEW checkpoint:", len(done_ids), "answers")
# else:
#     results, done_ids = [], set()
#     print("Starting a fresh run.")

# todo = base[~base["qa_id"].astype(str).isin(done_ids)].copy()
# print("Questions remaining:", len(todo), "of", len(base))

# start = time.time()
# for i, (_, row) in enumerate(todo.iterrows(), start=1):
#     try:
#         answer = generate_answer(row["question"], row["retrieved_context"])
#     except Exception as exc:
#         answer = ""
#         print("Generation error for", row["qa_id"], ":", repr(exc))

#     results.append({
#         "qa_id": str(row["qa_id"]),
#         "question": row["question"],
#         "generated_answer": answer
#     })

#     if i % 5 == 0 or i == len(todo):
#         checkpoint_df = pd.DataFrame(results).drop_duplicates("qa_id", keep="last")
#         checkpoint_df.to_csv(CHECKPOINT_PATH, index=False, encoding="utf-8-sig")
#         elapsed = (time.time() - start) / 60
#         print(f"Progress {i}/{len(todo)} this run | saved {len(checkpoint_df)} total | {elapsed:.1f} min")

# print("Generation loop finished.")
# print("Checkpoint:", CHECKPOINT_PATH)


In [13]:
# Cell 11 — Full Batch 180 Generation (Qwen2.5-7B RAG)
import time
from pathlib import Path
import pandas as pd
import torch

CHECKPOINT_PATH = "/kaggle/working/qwen7b_full_checkpoint.csv"
FINAL_OUTPUT_PATH = "/kaggle/working/qwen7b_full_results_180.csv"

if Path(CHECKPOINT_PATH).exists():
    checkpoint = pd.read_csv(CHECKPOINT_PATH, encoding="utf-8-sig", dtype={"qa_id": str})
    results = checkpoint.drop_duplicates("qa_id", keep="last").to_dict("records")
    done_ids = set(checkpoint["qa_id"].astype(str))
    print(f"Resuming from checkpoint: {len(done_ids)} questions already completed.")
else:
    results, done_ids = [], set()
    print("Starting fresh Qwen2.5-7B full batch run (180 questions)...")

todo = base[~base["qa_id"].astype(str).isin(done_ids)].copy()
print(f"Questions remaining: {len(todo)} of {len(base)}")

start_time = time.time()
for i, (_, row) in enumerate(todo.iterrows(), start=1):
    q_id = str(row["qa_id"])
    q_text = row["question"]
    ctx = row["retrieved_context"]
    
    try:
        rag_ans = generate_response(q_text, context=ctx, mode="rag")
    except Exception as e:
        rag_ans = f"Error: {e}"

    results.append({
        "qa_id": q_id,
        "question": q_text,
        "act_title": row["act_title"],
        "section_number": str(row["section_number"]),
        "final_answer": row["final_answer"],
        "qwen7b_answer": rag_ans
    })

    if i % 5 == 0 or i == len(todo):
        ckpt_df = pd.DataFrame(results).drop_duplicates("qa_id", keep="last")
        ckpt_df.to_csv(CHECKPOINT_PATH, index=False, encoding="utf-8-sig")
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            
        elapsed = (time.time() - start_time) / 60
        print(f"Progress: {i}/{len(todo)} | Saved: {len(ckpt_df)}/180 | Elapsed: {elapsed:.1f} mins")

final_df = pd.DataFrame(results).drop_duplicates("qa_id", keep="last")
final_df.to_csv(FINAL_OUTPUT_PATH, index=False, encoding="utf-8-sig")
print(f"✓ All 180 questions completed! Saved to: {FINAL_OUTPUT_PATH}")

Starting fresh Qwen2.5-7B full batch run (180 questions)...
Questions remaining: 180 of 180
Progress: 5/180 | Saved: 5/180 | Elapsed: 2.0 mins
Progress: 10/180 | Saved: 10/180 | Elapsed: 4.4 mins
Progress: 15/180 | Saved: 15/180 | Elapsed: 6.2 mins
Progress: 20/180 | Saved: 20/180 | Elapsed: 8.4 mins
Progress: 25/180 | Saved: 25/180 | Elapsed: 10.4 mins
Progress: 30/180 | Saved: 30/180 | Elapsed: 12.8 mins
Progress: 35/180 | Saved: 35/180 | Elapsed: 15.0 mins
Progress: 40/180 | Saved: 40/180 | Elapsed: 17.4 mins
Progress: 45/180 | Saved: 45/180 | Elapsed: 19.6 mins
Progress: 50/180 | Saved: 50/180 | Elapsed: 22.0 mins
Progress: 55/180 | Saved: 55/180 | Elapsed: 24.2 mins
Progress: 60/180 | Saved: 60/180 | Elapsed: 26.1 mins
Progress: 65/180 | Saved: 65/180 | Elapsed: 28.3 mins
Progress: 70/180 | Saved: 70/180 | Elapsed: 30.4 mins
Progress: 75/180 | Saved: 75/180 | Elapsed: 33.0 mins
Progress: 80/180 | Saved: 80/180 | Elapsed: 35.0 mins
Progress: 85/180 | Saved: 85/180 | Elapsed: 36.9 m

In [14]:
# # Cell 12 — Export a quality-control report for manual evaluation
# qc = final_df.copy()
# qc["answer_length"] = qc["generated_answer"].fillna("").str.len()
# qc["contains_replacement_char"] = qc["generated_answer"].fillna("").str.contains("�", regex=False)
# qc["looks_truncated"] = qc["generated_answer"].fillna("").str.strip().str.endswith(("�", "...", "…"))
# qc["blank"] = qc["generated_answer"].fillna("").str.strip().eq("")
# qc_path = "/kaggle/working/stage6_quality_control_report.csv"
# qc.to_csv(qc_path, index=False, encoding="utf-8-sig")
# print("QC report:", qc_path)
# print(qc[["answer_length", "contains_replacement_char", "looks_truncated", "blank"]].sum())
# print("Remember: these automatic checks do not detect all incorrect legal claims.")


In [15]:
# # Cell 12 — Automated Citation Verification & Trust Metrics (Fixed PROP-03)
# def norm_digits(s):
#     bn_digits = "০১২৩৪৫৬৭৮৯"
#     en_digits = "0123456789"
#     tr = str.maketrans(bn_digits, en_digits)
#     return str(s).translate(tr).strip().lower()

# df = pd.read_csv(FINAL_OUTPUT_PATH, encoding="utf-8-sig")

# rag_act_hits = 0
# rag_sec_hits = 0
# norag_act_hits = 0
# norag_sec_hits = 0
# total = len(df)

# for _, row in df.iterrows():
#     target_act = str(row["act_title"]).strip()
#     target_sec = norm_digits(str(row["section_number"]))
    
#     rag_ans = str(row["lexbanglarag_answer"])
#     norag_ans = str(row["norag_answer"])
    
#     if target_act in rag_ans: rag_act_hits += 1
#     if target_act in norag_ans: norag_act_hits += 1
        
#     if target_sec in norm_digits(rag_ans): rag_sec_hits += 1
#     if target_sec in norm_digits(norag_ans): norag_sec_hits += 1

# eval_summary = {
#     "Evaluation Metric": [
#         "Total Evaluated Questions",
#         "Act Citation Accuracy (%)",
#         "Section Citation Accuracy (%)",
#         "Abstention Count (Grounded Refusal)"
#     ],
#     "No-RAG Baseline": [
#         total,
#         f"{(norag_act_hits/total)*100:.2f}% ({norag_act_hits}/{total})",
#         f"{(norag_sec_hits/total)*100:.2f}% ({norag_sec_hits}/{total})",
#         0
#     ],
#     "LexBanglaRAG (Ours)": [
#         total,
#         f"{(rag_act_hits/total)*100:.2f}% ({rag_act_hits}/{total})",
#         f"{(rag_sec_hits/total)*100:.2f}% ({rag_sec_hits}/{total})",
#         int(df["lexbanglarag_answer"].str.contains("প্রদত্ত প্রমাণ থেকে নিশ্চিত উত্তর দেওয়া যাচ্ছে না").sum())
#     ]
# }

# eval_df = pd.DataFrame(eval_summary)
# display(eval_df)
# eval_df.to_csv("/kaggle/working/lexbanglarag_citation_evaluation_report.csv", index=False, encoding="utf-8-sig")

In [16]:
# Cell 12 — Qwen2.5-7B Smart Citation & F1 Benchmark
import unicodedata, re
import pandas as pd

def normalize_bn(text):
    if pd.isna(text): return ""
    text = unicodedata.normalize('NFC', str(text))
    bn_digits, en_digits = "০১২৩৪৫৬৭৮৯", "0123456789"
    text = text.translate(str.maketrans(bn_digits, en_digits))
    text = re.sub(r'[\[\]\(\)\,\.\-\–\—\:\;\"\'\’]', ' ', text)
    return re.sub(r'\s+', ' ', text).strip().lower()

def calculate_token_f1(pred, gold):
    pred_tokens = set(normalize_bn(pred).split())
    gold_tokens = set(normalize_bn(gold).split())
    if not pred_tokens or not gold_tokens:
        return 0.0
    common = pred_tokens.intersection(gold_tokens)
    if not common:
        return 0.0
    precision = len(common) / len(pred_tokens)
    recall = len(common) / len(gold_tokens)
    return (2 * precision * recall) / (precision + recall)

df = pd.read_csv("/kaggle/working/qwen7b_full_results_180.csv", encoding="utf-8-sig")

q7b_act_hits = 0
q7b_sec_hits = 0
q7b_f1_list = []
total = len(df)

for _, row in df.iterrows():
    raw_act = str(row["act_title"]).strip()
    norm_act = normalize_bn(raw_act)
    act_core = " ".join(norm_act.split()[:4])
    
    target_sec = normalize_bn(str(row["section_number"]))
    gold_ans = str(row["final_answer"])
    q7b_ans_norm = normalize_bn(row["qwen7b_answer"])
    
    if act_core in q7b_ans_norm or norm_act in q7b_ans_norm:
        q7b_act_hits += 1
        
    sec_regex = rf'\b{re.escape(target_sec)}\b'
    if re.search(sec_regex, q7b_ans_norm) or target_sec in q7b_ans_norm:
        q7b_sec_hits += 1
        
    q7b_f1_list.append(calculate_token_f1(row["qwen7b_answer"], gold_ans))

q7b_summary = {
    "Evaluation Metric": [
        "Total Evaluated Questions",
        "Act Citation Accuracy (%)",
        "Section Citation Accuracy (%)",
        "Average Answer Token F1-Score"
    ],
    "Qwen2.5-7B (4-bit)": [
        total,
        f"{(q7b_act_hits/total)*100:.2f}% ({q7b_act_hits}/{total})",
        f"{(q7b_sec_hits/total)*100:.2f}% ({q7b_sec_hits}/{total})",
        f"{sum(q7b_f1_list)/total:.4f}"
    ]
}

q7b_eval_df = pd.DataFrame(q7b_summary)
print("=== QWEN2.5-7B CITATION & ANSWER ACCURACY BENCHMARK ===")
display(q7b_eval_df)
q7b_eval_df.to_csv("/kaggle/working/qwen7b_evaluation_report.csv", index=False, encoding="utf-8-sig")

=== QWEN2.5-7B CITATION & ANSWER ACCURACY BENCHMARK ===


,Evaluation Metric,Qwen2.5-7B (4-bit)
0,Total Evaluated Questions,180
1,Act Citation Accuracy (%),40.56% (73/180)
2,Section Citation Accuracy (%),45.00% (81/180)
3,Average Answer Token F1-Score,0.3202


In [17]:
# # Cell 12 (Refined) — Smart Citation & Answer Similarity Benchmark
# import unicodedata, re
# import pandas as pd

# def normalize_bn(text):
#     if pd.isna(text): return ""
#     text = str(text)
#     # Unicode Normalization (NFC)
#     text = unicodedata.normalize('NFC', text)
#     # Bengali to English digits
#     bn_digits = "০১২৩৪৫৬৭৮৯"
#     en_digits = "0123456789"
#     tr = str.maketrans(bn_digits, en_digits)
#     text = text.translate(tr)
#     # Remove brackets, punctuation variations
#     text = re.sub(r'[\[\]\(\)\,\.\-\–\—\:\;\"\'\’]', ' ', text)
#     text = re.sub(r'\s+', ' ', text).strip().lower()
#     return text

# def calculate_token_f1(pred, gold):
#     pred_tokens = set(normalize_bn(pred).split())
#     gold_tokens = set(normalize_bn(gold).split())
#     if not pred_tokens or not gold_tokens:
#         return 0.0
#     common = pred_tokens.intersection(gold_tokens)
#     if not common:
#         return 0.0
#     precision = len(common) / len(pred_tokens)
#     recall = len(common) / len(gold_tokens)
#     return (2 * precision * recall) / (precision + recall)

# df = pd.read_csv("/kaggle/working/lexbanglarag_full_results_180.csv", encoding="utf-8-sig")

# rag_act_hits, norag_act_hits = 0, 0
# rag_sec_hits, norag_sec_hits = 0, 0
# rag_f1_list, norag_f1_list = [], []
# total = len(df)

# for _, row in df.iterrows():
#     raw_act = str(row["act_title"]).strip()
#     # Extract core Act name (e.g. without year if necessary)
#     norm_act = normalize_bn(raw_act)
#     # First 3-4 significant words of the Act
#     act_core = " ".join(norm_act.split()[:4])
    
#     target_sec = normalize_bn(str(row["section_number"]))
#     gold_ans = str(row["final_answer"])
    
#     rag_ans_norm = normalize_bn(row["lexbanglarag_answer"])
#     norag_ans_norm = normalize_bn(row["norag_answer"])
    
#     # Check Act Citation
#     if act_core in rag_ans_norm or norm_act in rag_ans_norm:
#         rag_act_hits += 1
#     if act_core in norag_ans_norm or norm_act in norag_ans_norm:
#         norag_act_hits += 1
        
#     # Check Section Citation
#     # Match standalone section number
#     sec_regex = rf'\b{re.escape(target_sec)}\b'
#     if re.search(sec_regex, rag_ans_norm) or target_sec in rag_ans_norm:
#         rag_sec_hits += 1
#     if re.search(sec_regex, norag_ans_norm) or target_sec in norag_ans_norm:
#         norag_sec_hits += 1
        
#     # Token F1 against Gold Answer
#     rag_f1_list.append(calculate_token_f1(row["lexbanglarag_answer"], gold_ans))
#     norag_f1_list.append(calculate_token_f1(row["norag_answer"], gold_ans))

# smart_summary = {
#     "Evaluation Metric": [
#         "Total Evaluated Questions",
#         "Act Citation Accuracy (%)",
#         "Section Citation Accuracy (%)",
#         "Average Answer Token F1-Score",
#         "F1 Gain over Baseline"
#     ],
#     "No-RAG Baseline": [
#         total,
#         f"{(norag_act_hits/total)*100:.2f}% ({norag_act_hits}/{total})",
#         f"{(norag_sec_hits/total)*100:.2f}% ({norag_sec_hits}/{total})",
#         f"{sum(norag_f1_list)/total:.4f}",
#         "Baseline (0.00%)"
#     ],
#     "LexBanglaRAG (Ours)": [
#         total,
#         f"{(rag_act_hits/total)*100:.2f}% ({rag_act_hits}/{total})",
#         f"{(rag_sec_hits/total)*100:.2f}% ({rag_sec_hits}/{total})",
#         f"{sum(rag_f1_list)/total:.4f}",
#         f"+{((sum(rag_f1_list)-sum(norag_f1_list))/sum(norag_f1_list))*100:.2f}%"
#     ]
# }

# smart_eval_df = pd.DataFrame(smart_summary)
# print("=== SMART CITATION & ANSWER ACCURACY BENCHMARK ===")
# display(smart_eval_df)
# smart_eval_df.to_csv("/kaggle/working/lexbanglarag_smart_evaluation_report.csv", index=False, encoding="utf-8-sig")

In [18]:
# # Cell 13 — Formal Hallucination Breakdown & Taxonomy Table
# import pandas as pd
# import unicodedata, re

# def normalize_bn(text):
#     if pd.isna(text): return ""
#     text = unicodedata.normalize('NFC', str(text))
#     bn_digits, en_digits = "০১২৩৪৫৬৭৮৯", "0123456789"
#     text = text.translate(str.maketrans(bn_digits, en_digits))
#     text = re.sub(r'[\[\]\(\)\,\.\-\–\—\:\;\"\'\’]', ' ', text)
#     return re.sub(r'\s+', ' ', text).strip().lower()

# df = pd.read_csv("/kaggle/working/lexbanglarag_full_results_180.csv", encoding="utf-8-sig")

# def categorize_hallucination(ans, target_act, target_sec):
#     ans_norm = normalize_bn(ans)
#     act_core = " ".join(normalize_bn(target_act).split()[:4])
#     sec_norm = normalize_bn(str(target_sec))
    
#     # 1. Check Abstention
#     if "প্রদত্ত প্রমাণ থেকে নিশ্চিত উত্তর দেওয়া যাচ্ছে না" in ans or "নিশ্চিতভাবে আপনার প্রশ্নের জবাব দিতে পারি না" in ans:
#         return "Appropriate Abstention"
    
#     act_match = act_core in ans_norm
#     sec_match = sec_norm in ans_norm
    
#     # 2. Both correct
#     if act_match and sec_match:
#         return "Fully Grounded (Faithful)"
#     # 3. Only Act or Section correct
#     elif act_match or sec_match:
#         return "Partially Grounded (Minor Misattribution)"
#     # 4. Neither matched -> Pure Hallucination / Fabrication
#     else:
#         return "Fabricated / Hallucinated"

# rag_cats = df.apply(lambda r: categorize_hallucination(r["lexbanglarag_answer"], r["act_title"], r["section_number"]), axis=1)
# norag_cats = df.apply(lambda r: categorize_hallucination(r["norag_answer"], r["act_title"], r["section_number"]), axis=1)

# tax_df = pd.DataFrame({
#     "No-RAG Count": norag_cats.value_counts(),
#     "No-RAG (%)": (norag_cats.value_counts(normalize=True) * 100).round(2),
#     "LexBanglaRAG Count": rag_cats.value_counts(),
#     "LexBanglaRAG (%)": (rag_cats.value_counts(normalize=True) * 100).round(2)
# }).fillna(0)

# print("=== TABLE: FORMAL HALLUCINATION & FAITHFULNESS TAXONOMY ===")
# display(tax_df)
# tax_df.to_csv("/kaggle/working/lexbanglarag_hallucination_taxonomy.csv", encoding="utf-8-sig")

## Output files
- `stage6_pilot_5_answers.csv` — 5-question pilot
- `stage6_clean_generation_checkpoint.csv` — checkpoint for resuming the run
- `stage6_generated_answers_clean_v1.csv` — final generated answers, separate from the gold QA
- `stage6_quality_control_report.csv` — simple quality flags

If the session disconnects, reopen this notebook and run the setup/load/model cells again, then run Cell 10. It resumes from the **new checkpoint** if that file is still present in the same Kaggle session. If Kaggle's working storage was reset, upload/download the checkpoint as a Kaggle dataset input to preserve it.
